# Method: the bootstrap

*Question → Intuition → Math → Code → Assumptions → How it breaks*

## 1. Question

The ranking says one player scored 2.94 and another 2.81. Is that a difference,
or is it noise?

Every number in the table is computed from a finite career — five seasons,
nineteen seasons — and a different set of seasons would have produced a slightly
different number. The question is how much slightly.

## 2. Intuition

If we could re-run a career a thousand times we would see how much the average
bounces around. We cannot. We have the seasons that happened.

The bootstrap's trick is to treat the seasons we have **as if they were the
population**, and draw new careers from them — same length, sampling with
replacement, so some seasons appear twice and others not at all. Each fake career
gives a mean. The spread of those means estimates the spread of the real one.

It sounds like cheating. It works because the sample is our best available
picture of the population, and resampling from it mimics the sampling that
produced it in the first place.

## 3. Math

Given observations $x_1 \dots x_n$, for $b = 1 \dots B$:

1. Draw $n$ values from $\{x_i\}$ **with replacement**, giving $x^*_{1} \dots x^*_{n}$
2. Record $\bar{x}^*_b$, the mean of that draw

The 95% percentile interval is the 2.5th and 97.5th percentiles of
$\{\bar{x}^*_1 \dots \bar{x}^*_B\}$.

**With replacement is the whole method.** Sampling without replacement from $n$
observations returns the same $n$ observations every time and the interval would
be zero-width.

We use $B = 10{,}000$. The percentile interval is the simplest of several
variants; BCa corrects for skew and bias, and would be the upgrade if the
intervals were being used for formal inference rather than for saying "these two
players are not separable".

In [ ]:
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.figsize"] = (10, 5.5)

SAMPLE = "../data/sample"
ranking = pd.read_parquet(f"{SAMPLE}/ranking.parquet")
seasons = pd.read_parquet(f"{SAMPLE}/player_season_scored.parquet")
offsets = pd.read_parquet(f"{SAMPLE}/league_offsets.parquet")

In [ ]:
from gambeta import doubt

career = seasons[seasons["player"] == "Thierry Henry"]["season_score"].to_numpy()
print(f"{len(career)} seasons: {np.round(career, 2)}")

est, lo, hi = doubt.bootstrap(career, n=10_000)
print(f"\nmean {est:.3f}, 95% interval [{lo:.3f}, {hi:.3f}], width {hi - lo:.3f}")

Watch the machinery run, so it is not a black box.

In [ ]:
rng = np.random.default_rng(20260810)
draws = rng.choice(career, size=(10_000, len(career)), replace=True).mean(axis=1)

fig, ax = plt.subplots()
ax.hist(draws, bins=60, color="#4C78A8", edgecolor="none")
for value, style, label in [
    (career.mean(), "-", "observed mean"),
    (np.percentile(draws, 2.5), "--", "2.5th / 97.5th percentile"),
    (np.percentile(draws, 97.5), "--", None),
]:
    ax.axvline(value, color="#333", linestyle=style, linewidth=1.2, label=label)
ax.set_xlabel("mean of a resampled career")
ax.set_ylabel("count")
ax.set_title("10,000 careers that could have happened")
ax.legend()
plt.show()

## The interval narrows with evidence, not with effort

More seasons, more confidence. This is the one property worth checking by hand
before trusting any interval.

In [ ]:
lengths = []
for n in (3, 5, 8, 12, 18):
    pool = seasons.groupby("player")["season_score"].agg(["count", "mean"])
    who = pool[pool["count"] == n]
    if not len(who):
        continue
    name = who["mean"].idxmax()
    values = seasons[seasons["player"] == name]["season_score"].to_numpy()
    mid, low, high = doubt.bootstrap(values, n=10_000)
    lengths.append(
        {"seasons": n, "player": name, "est": mid, "lo": low, "hi": high, "width": high - low}
    )

pd.DataFrame(lengths).round(3)

## The finding this produces

The reason the project bootstraps at all is not decoration. It is that the
intervals turn out to **overlap almost everywhere at the top of the table**, and
that changes what can honestly be claimed.

In [ ]:
top = ranking[ranking["qualified"]].head(10)["player"].tolist()
bands = {}
for name in top:
    values = seasons[seasons["player"] == name]["season_score"].to_numpy()
    bands[name] = doubt.bootstrap(values, n=10_000)

pairs = [(a, b) for i, a in enumerate(top) for b in top[i + 1 :]]
overlapping = sum(1 for a, b in pairs if bands[a][1] <= bands[b][2] and bands[b][1] <= bands[a][2])
print(f"{overlapping} of {len(pairs)} pairs in the top ten have overlapping intervals")
print("\nSeparable pairs:")
for a, b in pairs:
    if not (bands[a][1] <= bands[b][2] and bands[b][1] <= bands[a][2]):
        print(f"  {a} vs {b}")

## 5. Assumptions

1. **The observations are independent.** Resampling seasons assumes one season
   tells you nothing about the next. For a career this is questionable — form,
   age and club quality all persist — and dependence makes the true interval
   *wider* than the bootstrap reports.
2. **The sample represents the population.** With five seasons it represents it
   badly, which the interval width reflects honestly.
3. **The statistic is smooth.** The bootstrap works well for means and poorly for
   maxima and other quantities dominated by a single extreme observation.

## 6. How it breaks

**One observation gives perfect confidence.** This is not hypothetical — it
shipped, and it is recorded in `DEVIATIONS.md` #5.

In [ ]:
print("a one-season 'career':", doubt.bootstrap(np.array([3.4]), n=10_000))
print("\nEvery resample of one number is that number, so the interval has zero")
print("width -- and renders as perfect certainty about the least certain")
print("estimate on the page. gambeta's answer is a minimum-seasons floor, not a")
print("cleverer interval: the problem is the question, not the arithmetic.")

**And the subtler failure: a narrow interval is not a correct one.**
The bootstrap quantifies sampling variability and nothing else. If the metric is
biased — if it systematically undervalues defenders, say — every resample carries
the same bias and the interval sits confidently around the wrong number.

An interval says *"if the world worked the way this model assumes, this is how
much the answer would wobble."* It never says the model was right.